In [4]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

In [5]:
import pandas as pd
import torch
import json

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [6]:
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

train_df = load_jsonl("Train_validation_Merged_ASQE_N4000.jsonl")
test_df = load_jsonl("DEV_TEST_Merged_ASQE_N2500.jsonl")

train_df.head()

,text,labels
0,He is not helpful at all and makes fun of stud...,"[{'aspect': 'He', 'opinion': 'Very rude', 'pol..."
1,"This course was amazing. For the first time, t...","[{'aspect': 'This course', 'opinion': 'I loved..."
2,Exams were very easy and he allowed us to do e...,"[{'aspect': 'Exams', 'opinion': 'very easy', '..."
3,The lecturers from Exeter have helped me throu...,"[{'aspect': 'lecturers', 'opinion': 'helped me..."
4,"Extremely boring and lectures were useless, bu...","[{'aspect': 'lectures', 'opinion': 'Extremely ..."


In [7]:
def convert_to_absa(df):
    rows = []
    
    for _, row in df.iterrows():
        sentence = row["text"]
        label_list = row["labels"]
        
        for item in label_list:
            aspect = item.get("aspect")
            sentiment = item.get("polarity")
            
            if aspect is None or aspect == "null":
                continue
            
            rows.append({
                "sentence": sentence,
                "aspect": aspect,
                "sentiment": sentiment
            })
    
    df_new = pd.DataFrame(rows)
    df_new = df_new.drop_duplicates().reset_index(drop=True)
    
    return df_new

train_absa = convert_to_absa(train_df)
test_absa = convert_to_absa(test_df)

train_absa["sentiment"] = train_absa["sentiment"].str.capitalize()
test_absa["sentiment"] = test_absa["sentiment"].str.capitalize()

print("Train:", train_absa.shape)
print("Test:", test_absa.shape)

test_absa.head()

Train: (10331, 3)
Test: (6600, 3)


,sentence,aspect,sentiment
0,She will only accept. docx forms in microsoft ...,This professor,Negative
1,Great Teacher! Learned a lot! If coming to cla...,Teacher,Positive
2,Not so easy class; great instructor!,class,Negative
3,Not so easy class; great instructor!,instructor,Positive
4,Studied for the exam literally the day of... e...,textbook,Negative


In [8]:
def create_prompt(row):
    return f"""<s>[INST]
You are an expert in aspect-based sentiment analysis.

Task:
Classify the sentiment of the given aspect in the sentence.

Rules:
- Output ONLY one word: Positive, Neutral, or Negative
- Do NOT explain

Sentence: {row['sentence']}
Aspect: {row['aspect']}
[/INST]
{row['sentiment']}"""

train_absa["text"] = train_absa.apply(create_prompt, axis=1)
test_absa["text"] = test_absa.apply(create_prompt, axis=1)

In [9]:
print(train_absa["text"].iloc[0])

<s>[INST]
You are an expert in aspect-based sentiment analysis.

Task:
Classify the sentiment of the given aspect in the sentence.

Rules:
- Output ONLY one word: Positive, Neutral, or Negative
- Do NOT explain

Sentence: He is not helpful at all and makes fun of students when they ask questions. Very rude!!!
Aspect: He
[/INST]
Negative


In [10]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_absa[["text"]])
test_dataset = Dataset.from_pandas(test_absa[["text"]])

In [11]:
print(train_dataset[0])

{'text': '<s>[INST]\nYou are an expert in aspect-based sentiment analysis.\n\nTask:\nClassify the sentiment of the given aspect in the sentence.\n\nRules:\n- Output ONLY one word: Positive, Neutral, or Negative\n- Do NOT explain\n\nSentence: He is not helpful at all and makes fun of students when they ask questions. Very rude!!!\nAspect: He\n[/INST]\nNegative'}


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

# Quantization config (NEW WAY)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,   # ✅ NEW
    device_map="auto"
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded successfully


In [14]:
import torch
print(torch.cuda.is_available())

True


In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # 🔥 critical
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [25]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama_lora_results",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    num_train_epochs=2,
    learning_rate=2e-4,

    fp16=False,        
    bf16=False,       

    logging_steps=50,
    save_strategy="epoch",

    report_to="none"
)

In [26]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/10331 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10331 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10331 [00:00<?, ? examples/s]

In [27]:
trainer.train()

Step,Training Loss
50,1.913940
100,1.592002
150,1.516180
200,1.566910
250,1.519811
300,1.506727
350,1.500424
400,1.466612
450,1.440158
500,1.448647


TrainOutput(global_step=2584, training_loss=1.2347242750988656, metrics={'train_runtime': 3840.3178, 'train_samples_per_second': 5.38, 'train_steps_per_second': 0.673, 'total_flos': 1.4342113394085888e+17, 'train_loss': 1.2347242750988656})

In [28]:
def predict(sentence, aspect):
    prompt = f"""<s>[INST]
Classify the sentiment of the given aspect.

Sentence: {sentence}
Aspect: {aspect}
[/INST]
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = response.split("[/INST]")[-1].strip()

  
    if "Positive" in pred:
        return "Positive"
    elif "Negative" in pred:
        return "Negative"
    elif "Neutral" in pred:
        return "Neutral"
    else:
        return "Neutral"

In [29]:
predictions = []

for i, row in test_absa.iterrows():
    pred = predict(row["sentence"], row["aspect"])
    predictions.append(pred)

    if i % 200 == 0:
        print(f"Processed {i}/{len(test_absa)}")

test_absa["llama_pred"] = predictions

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processed 0/6600


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Processed 200/6600


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

...

Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `ma

Processed 6200/6600


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Processed 6400/6600


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=5) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

In [30]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(test_absa["sentiment"], test_absa["llama_pred"])
f1 = f1_score(test_absa["sentiment"], test_absa["llama_pred"], average="weighted")

print("LLaMA + LoRA Accuracy:", accuracy)
print("LLaMA + LoRA F1:", f1)

print(classification_report(
    test_absa["sentiment"],
    test_absa["llama_pred"]
))

LLaMA + LoRA Accuracy: 0.6937878787878788
LLaMA + LoRA F1: 0.7269371344360562
              precision    recall  f1-score   support

    Negative       0.82      0.74      0.77      2246
     Neutral       0.25      0.54      0.34       845
    Positive       0.90      0.70      0.79      3509

    accuracy                           0.69      6600
   macro avg       0.66      0.66      0.64      6600
weighted avg       0.79      0.69      0.73      6600

